# Data generation

The main problematic of this project is data:
- Where do we get data from that match our needs:
  - Project description.
  - Internship summary.
  - extracurricular activity overview.
- Labelling and filtring this data to our specific categories:
  - technologies.
  - domains.


Concerns: 
- What if a user wants to create a list in his description with special characters.
> Solution: descriptions should be normalised in the preprocessing step before passing them into the model for predictions.
- What if the description doesn't contain any technologies, but the user still wants them in his project labels.
> The model itself can't handle this situation. But he needs to be ready for it to avoid hallucinations, so the dataset needs to have descriptions with no technologies stated. And the model needs to be able to output an empty technologies list. But later we need to fix this problem by checking if technologies found is empty or less than a certain number, to add suggested technologies depending on the domains detected in the description in a confidence score order.
- what if the project reffers to different domains.
> The dataset should have multi-labels examples, and the architecture of the models classification should support multi-labels and overlapping domains, with a certain confidence score.


In [ ]:
import anthropic
import json
import time
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, JSON

load_dotenv()

True

# 1. Manual Data Collection and Preparation

Starting directly with synthetic data can lead to limited robustness and poor alignment with real-world text distributions. For this reason, we prioritized collecting real-world examples as a first step in the dataset creation process.

## Data Collection Pipeline

The dataset was built using a semi-manual pipeline designed to ensure both quality and relevance of the training data. The process followed these steps:

- Collecting real-world text samples (project descriptions, certifications, and internship experiences) from LinkedIn profiles
- Organizing the extracted information (title, description, and type) into a structured CSV file
- Converting the CSV file into a JSON format using a Python preprocessing script
- Importing the JSON dataset into Label Studio for annotation and classification
- Exporting the annotated data in raw JSONL format, including metadata generated during labeling
- Cleaning and standardizing the exported JSONL using a Python script, keeping only the fields required for model training

# 2. Data Augmentation using a LLM

>Synthetic data generation offers a powerful solution to this problematic, unlike manual data creation methods, using LLMs allows for the generation of rich, nuanced, and contextually relevant datasets that can significantly enhance it’s usefulness to enterprises and developers. *from [Dylan Royan Almeida](https://developers.openai.com/cookbook/examples/sdg1/)*

On top of that generating synthetic data manually will be time consuming and costly.

But using LLMs as data generators still raises some concerns:
- accuracy (does the data make sense).
- consistency (are two separate data points for the same input roughly the same).
- diversity (making sure our data distribution is balenced).

# 3. Data validation

In this step we created a script to validate our dataset on the following rules:
- Missing values
- Valid start and end entities
- no overlapping entities
- domains belong to allowed list only
- domains balence 

In [30]:
# choosing the path and the allowed labels

import json
from pathlib import Path

DATASET_PATH = "../data/raw/dataset_fixed.jsonl"

ALLOWED_DOMAIN_LABELS = {
    "Web Frontend",
    "Web Backend",
    "Mobile Development",
    "DevOps and Cloud Infrastructure",
    "Data Engineering",
    "Machine Learning and AI",
    "Cybersecurity",
    "Embedded Systems and IoT",
    "High Performance and Quantum Computing",
}

Here is an example of what our data looks like :

In [8]:
with open(DATASET_PATH,"r",encoding="utf-8") as f:
    first_line = f.readline().strip()

print(first_line)

{"text": "Design and Dev Collab Certificate. Awarded the 'Design and Dev Collab' certificate for participating in a collaborative hackathon focused on bridging UI/UX design and frontend development. Contributed as a Frontend Developer, implementing responsive interfaces using React and modern web technologies while collaborating closely with designers to deliver a functional and user-centered solution under tight deadlines.", "language": "en", "entities": [{"start": 267, "end": 272, "label": "TECH", "value": "React"}], "domain_labels": ["Web Frontend"]}


In [12]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())

print(first_line)

{'text': "Design and Dev Collab Certificate. Awarded the 'Design and Dev Collab' certificate for participating in a collaborative hackathon focused on bridging UI/UX design and frontend development. Contributed as a Frontend Developer, implementing responsive interfaces using React and modern web technologies while collaborating closely with designers to deliver a functional and user-centered solution under tight deadlines.", 'language': 'en', 'entities': [{'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}], 'domain_labels': ['Web Frontend']}


### Here we will give some examples of checks we will apply to our dataset and how they work

In [ ]:
def validate_existence(sample):
    is_valid = True # if this stays true that means the sample passed all tests
    # Validate if text is a string
    text = sample.get("text")
    if not isinstance(text,str) or not text.strip():
        print(f"This line is not a text or only has spaces")
        is_valid = False

    if(is_valid):
        print("Text is correct!")

validate_existence(first_line)
validate_existence({'text': "  ", 'language': 'en', 'entities': [{'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}], 'domain_labels': ['Web Frontend']})

Text is correct!
This line is not a text or only has spaces


In [34]:
def validate_type(sample):
    entities = sample.get("entities",[])
    if not isinstance(entities,list):
        print("entities must be a list")
    else:
        for entity in entities:
            if not isinstance(entity,dict):
                print("entities should contain a list of dicts")
        print("correct")
    
validate_type(first_line)
validate_type({'text': "", 'language': 'en', 'entities': {'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}, 'domain_labels': ['Web Frontend']})


correct
entities must be a list


In [32]:
def validate_domain_labels(sample):
    domain_labels = sample.get("domain_labels", [])
    is_valid=True
    if not isinstance(domain_labels,list):
        print("domain labels must be a list")
        is_valid=False
    else:
        for domain in domain_labels:
            if domain not in ALLOWED_DOMAIN_LABELS:
                print(f"invalid domain label :{domain}")
                is_valid = False
    if(is_valid):
        print("correct")

validate_domain_labels(first_line)
validate_domain_labels({'text': "", 'language': 'en', 'entities': {'start': 267, 'end': 272, 'label': 'TECH', 'value': 'React'}, 'domain_labels': ['TEST']})

correct
invalid domain label :TEST


Other cheks are made inside each entity to check if they are:
- empty
- their type is an int if its (start or end) and its an str if its (value)
- start < end
- start > 0 , end < len(text)
- exact substring match (text[start:end] == value)
- check for duplicate spans
- overlapping entities

### After running the `validate_jsonl.py` script this was our output:


VALIDATION SUMMARY

`Total samples:   247`

`Valid samples:   101`

`Invalid samples: 146`

which means we have 146 samples to clean. speciffically due to the use of an LLM (ChatGpt5) for generating the last few samples, it caused multiple imprecisions .
The faster way to solve the problem is to send the `validate_output_V1.txt` and the `datase_fixed.jsonl` to CLOUD and let it fix it one by one.

After the fix we have 247 clean samples with 0 invalid inputs.
